In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import linregress

### Cluster relation

In [2]:
df_full = pd.read_csv("img_attribute.csv")
df_full.rename(columns={'Absolute_Humidity(g/m^3)': 'AbsHumidity'}, inplace=True)

In [3]:
colors = [
    '#FF4B4B', '#FF9600', '#FFD24B', '#AFE182',
    '#82AFE1', '#AF96FF', '#969696', '#555555',
    '#000000', '#0066CC'
]
features = [
    'Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted', 'Stratocumulus_weighted',
    'Cumulus_weighted', 'Cirrocumulus_weighted', 'Nimbus_weighted', 'TotalCloud_weighted',
    'Clear_weighted', 'AbsHumidity'
]
cloud_features = [
    'Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted', 'Stratocumulus_weighted',
    'Cumulus_weighted', 'Cirrocumulus_weighted', 'Nimbus_weighted', 'TotalCloud_weighted', 'Clear_weighted'
]
target = 'Radiation Value'

clusters = [3, 2, 1]

feature_titles = [f.replace('_weighted', '') for f in features]
cluster_titles = [f"Cluster {c}" for c in clusters]

fig = make_subplots(
    rows=10, 
    cols=3,
    subplot_titles=cluster_titles,
    vertical_spacing=0.03,
    horizontal_spacing=0.07
)

for row_idx, feature in enumerate(features):
    for col_idx, cluster in enumerate(clusters):
        row = row_idx + 1
        col = col_idx + 1
        
        # filter data for the cluster
        df = df_full[df_full['label'] == cluster].copy()
        df['TotalCloud_weighted'] = df[['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted',
                                       'Stratocumulus_weighted', 'Cumulus_weighted', 'Cirrocumulus_weighted',
                                       'Nimbus_weighted']].sum(axis=1)
        
        # filter data where feature value >= 0.01
        df_plot = df[df[feature] >= 0.01]
        
        fig.add_trace(
            go.Scatter(
                x=df_plot[feature],
                y=df_plot[target],
                mode='markers',
                marker=dict(
                    size=5,
                    opacity=0.8,
                    color=colors[row_idx % len(colors)]
                )
            ),
            row=row,
            col=col
        )
        
        # if data > 5% then add regression line
        if not df_plot.empty and len(df_plot) / len(df) > 0.05:
            slope, intercept, r_value, p_value, std_err = linregress(df_plot[feature], df_plot[target])
            
            x_range = np.array([df_plot[feature].min(), df_plot[feature].max()])
            y_range = slope * x_range + intercept
            
            fig.add_trace(
                go.Scatter(
                    x=x_range,
                    y=y_range,
                    mode='lines',
                    line=dict(color='dimgrey', width=4)
                ),
                row=row,
                col=col
            )
            
            fig.add_annotation(
                x=0.97, y=0.05,
                xref="x domain", yref="y domain",
                text=f"Slope: {slope:.2f}, p: {p_value:.2f}",
                showarrow=False,
                row=row, col=col,
                align='right',
                font=dict(size=24, family='Arial', weight='bold'),
                bgcolor="rgba(255, 255, 255, 0.5)"
            )

        if feature in cloud_features:
            fig.update_xaxes(range=[-0.1, 1.1], row=row, col=col)
        elif feature == 'Absolute_Humidity(g/m^3)':
            fig.update_xaxes(range=[5, 25], row=row, col=col)
        
        if col == 1:
            fig.update_yaxes(title_text=feature_titles[row_idx], row=row, col=col,
                             title_font=dict(size=30, family='Arial', weight='bold'))

fig.update_yaxes(range=[-59, 10])
fig.update_layout(
    height=2000,
    width=1000,
    showlegend=False,
    margin=dict(l=40, r=10, t=50, b=30),
    template='simple_white',
)
fig.update_xaxes(tickfont=dict(size=24, family='Arial', weight='bold'), linewidth=3, linecolor='black')
fig.update_yaxes(tickfont=dict(size=24, family='Arial', weight='bold'), linewidth=3, linecolor='black')

for annotation in fig['layout']['annotations']:
    if annotation['text'] in cluster_titles:
        annotation['font'] = dict(size=30, family='Arial', weight='bold')
        annotation['y'] += 0.005

fig.show()

In [4]:
colors = ['#AFE182','#0066CC','#555555']
features = ['Stratocumulus_weighted', 'AbsHumidity', 'TotalCloud_weighted']
cloud_features = ['Stratocumulus_weighted', 'TotalCloud_weighted']
target = 'Radiation Value'

clusters = [3, 2, 1]

feature_titles = [f.replace('_weighted', '') for f in features]
cluster_titles = [f"Cluster {c}" for c in clusters]

fig = make_subplots(
    rows=3, 
    cols=3,
    subplot_titles=cluster_titles,
    vertical_spacing=0.1,
    horizontal_spacing=0.07
)

for row_idx, feature in enumerate(features):
    for col_idx, cluster in enumerate(clusters):
        row = row_idx + 1
        col = col_idx + 1
        
        # filter data for the cluster
        df = df_full[df_full['label'] == cluster].copy()
        df['TotalCloud_weighted'] = df[['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted',
                                       'Stratocumulus_weighted', 'Cumulus_weighted', 'Cirrocumulus_weighted',
                                       'Nimbus_weighted']].sum(axis=1)
        
        # filter data where feature value >= 0.01
        df_plot = df[df[feature] >= 0.01]
        
        fig.add_trace(
            go.Scatter(
                x=df_plot[feature],
                y=df_plot[target],
                mode='markers',
                marker=dict(
                    size=5,
                    opacity=0.8,
                    color=colors[row_idx % len(colors)]
                )
            ),
            row=row,
            col=col
        )
        
        # if data > 5% then add regression line
        if not df_plot.empty and len(df_plot) / len(df) > 0.05:
            slope, intercept, r_value, p_value, std_err = linregress(df_plot[feature], df_plot[target])
            
            x_range = np.array([df_plot[feature].min(), df_plot[feature].max()])
            y_range = slope * x_range + intercept
            
            fig.add_trace(
                go.Scatter(
                    x=x_range,
                    y=y_range,
                    mode='lines',
                    line=dict(color='dimgrey', width=4)
                ),
                row=row,
                col=col
            )
            
            fig.add_annotation(
                x=0.97, y=0.05,
                xref="x domain", yref="y domain",
                text=f"Slope: {slope:.2f}, p: {p_value:.2f}",
                showarrow=False,
                row=row, col=col,
                align='right',
                font=dict(size=24, family='Arial', weight='bold'),
                bgcolor="rgba(255, 255, 255, 0.5)"
            )

        if feature in cloud_features:
            fig.update_xaxes(range=[-0.1, 1.1], row=row, col=col)
        elif feature == 'Absolute_Humidity(g/m^3)':
            fig.update_xaxes(range=[5, 25], row=row, col=col)
        
        if col == 1:
            fig.update_yaxes(title_text=feature_titles[row_idx], row=row, col=col,
                             title_font=dict(size=30, family='Arial', weight='bold'))

fig.update_yaxes(range=[-59, 10])
fig.update_layout(
    height=700,
    width=1000,
    showlegend=False,
    margin=dict(l=40, r=10, t=40, b=30),
    template='simple_white'
)
fig.update_xaxes(tickfont=dict(size=24, family='Arial', weight='bold'), linewidth=3, linecolor='black')
fig.update_yaxes(tickfont=dict(size=24, family='Arial', weight='bold'), linewidth=3, linecolor='black')

for annotation in fig['layout']['annotations']:
    if annotation['text'] in cluster_titles:
        annotation['font'] = dict(size=30, family='Arial', weight='bold')
        annotation['y'] += 0.005

fig.show()

### Cluster composition

In [5]:
df = pd.read_csv("img_attribute.csv")

cloud_features = [
    'Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted', 'Stratocumulus_weighted',
    'Cumulus_weighted', 'Cirrocumulus_weighted', 'Nimbus_weighted', 'Clear_weighted'
]

cluster_cloud = df.groupby('label')[cloud_features].sum()
clusters_ordered = [1, 2, 3]
cluster_cloud = cluster_cloud.reindex(clusters_ordered)

colors = [
    '#FF4B4B', '#FF9600', '#FFD24B',
    '#AFE182', '#82AFE1', '#AF96FF',
    '#969696', '#000000' 
]

In [6]:
# bar chart
fig = go.Figure()

for i, feature in enumerate(cloud_features):
    fig.add_trace(go.Bar(
        y=[f'{label} ' for label in cluster_cloud.index],
        x=cluster_cloud[feature],
        name=feature.replace('_weighted', ''),
        orientation='h',
        marker=dict(color=colors[i]),
        showlegend=False
    ))

for cloud_type, color in zip(reversed(cloud_features), reversed(colors)):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=60, color=color, symbol="square"),
        name=cloud_type.replace('_weighted', ''), showlegend=True, hoverinfo='skip'
    ))

fig.update_layout(
    barmode='stack',
    barnorm='percent',
    xaxis_title='Proportion',
    yaxis_title='Cluster',
    yaxis=dict(type='category'),
    # legend_traceorder='normal',
    height=350, width=1600,
    font=dict(size=30, family="Arial", weight="bold"),
    margin=dict(l=100, r=10, t=50, b=80),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.05,
        xanchor="center", x=0.5
    )
)

fig.show()

In [7]:
# pie chart
fig = make_subplots(rows=1, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

for i, cluster_label in enumerate(clusters_ordered):
    cluster_data = cluster_cloud.loc[cluster_label]
    
    labels = [name.replace('_weighted', '') for name in cloud_features]
    values = [cluster_data[feature] for feature in cloud_features]
    
    fig.add_trace(go.Pie(
        labels=labels, 
        values=values,
        name=f'Cluster {cluster_label}',
        marker=dict(colors=colors),
        hoverinfo='label+value',
        textinfo='none',
        showlegend=False,
        sort=False
    ), 1, i+1)
    
    fig.add_annotation(
        text=f'Cluster {cluster_label}',
        x=0.1 + i*0.4,
        y=0.96,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=24, family="Arial", weight="bold")
    )

for cloud_type, color in zip(cloud_features, colors):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=20, color=color, symbol="square"),
        name=cloud_type.replace('_weighted', ''), showlegend=True, hoverinfo='skip'
    ))

fig.update_layout(
    height=500, width=1200,
    font=dict(size=22, family="Arial", weight="bold"),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=0.03,
        xanchor="center", x=0.5
    ),
    margin=dict(l=20, r=20, t=0, b=0),
    template='simple_white',
)

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

fig.show()